# Judy Neural-Link

Runtime → **T4 GPU**. Paste the launcher session code into the form, run the cell, and leave it open.

Models download from Hugging Face. The tunnel is Cloudflare. No Drive, no ngrok.

In [ ]:
HF_REPO = "wolfdead/judy-neural-link-voice"  # @param {type:"string"}
HF_REVISION = "main"  # @param {type:"string"}
SESSION_CODE = ""  # @param {type:"string"}

import os, sys, time, subprocess, threading, urllib.request

HF_REPO = HF_REPO.strip()
HF_REVISION = HF_REVISION.strip() or "main"
SESSION_CODE = SESSION_CODE.strip()
if "YOUR_HF_USERNAME" in HF_REPO or "/" not in HF_REPO:
    raise SystemExit("Set HF_REPO to username/judy-neural-link-voice.")
if len(SESSION_CODE) < 12:
    raise SystemExit("Paste the launcher session code into SESSION_CODE.")
if HF_REVISION == "main":
    print("Note: HF_REVISION is 'main' (latest). Releases should pin a commit SHA.")

print("[1/4] Packages")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "coqui-tts==0.27.5", "transformers==4.57.6", "faiss-cpu",
    "librosa", "soundfile", "flask", "huggingface_hub",
])

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU found. Runtime > Change runtime type > T4 GPU, then run again.")

print("[2/4] Hugging Face:", HF_REPO, "@", HF_REVISION)
from huggingface_hub import snapshot_download
root = snapshot_download(repo_id=HF_REPO, revision=HF_REVISION)
os.chdir(root)
sys.path.insert(0, root)
os.environ["COQUI_TOS_AGREED"] = "1"
os.environ["JUDY_TOKEN"] = SESSION_CODE

print("[3/4] Loading voice models onto the GPU")
from colab_server import app
import relay

def _serve():
    app.run(host="127.0.0.1", port=5000, threaded=True, use_reloader=False)

threading.Thread(target=_serve, daemon=True).start()
for _ in range(40):  # wait until the local server answers
    try:
        urllib.request.urlopen("http://127.0.0.1:5000/health", timeout=2).read()
        break
    except Exception:
        time.sleep(0.5)

print("[4/4] Cloudflare tunnel")
bin_path = "/content/cloudflared"
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    bin_path,
)
os.chmod(bin_path, 0o755)
tunnel = subprocess.Popen(
    [bin_path, "tunnel", "--url", "http://127.0.0.1:5000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public = None
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    print(line, end="")
    found = relay.URL_FIND_RE.search(line)
    if found:
        public = found.group(0)
        break
if not public:
    raise SystemExit("Cloudflare did not return a link.")

# Keep draining cloudflared's output, otherwise its pipe fills up and the tunnel stalls.
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

relay.publish(SESSION_CODE, public)
print("\n=== CONNECTED ===")
print(public)
print("Enter the game when the launcher light turns green. Leave this cell running.")

last_publish = time.time()
while tunnel.poll() is None:
    time.sleep(15)
    if time.time() - last_publish > 1800:  # the relay forgets old messages, so refresh the link
        try:
            relay.publish(SESSION_CODE, public)
            last_publish = time.time()
        except Exception as e:
            print("Relay refresh failed:", e)
print("Tunnel closed. Run the cell again to reconnect.")
